# Publication notes
See [the project README](../README.md) and [setup instructions](../docs/setup.md). Outputs were cleared for publication; this file has not been rerun during packaging. Notebook 6 is the current integration development reference; supplied-run evidence and limitations are documented in the README.


In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

from pydrake.all import QuasidynamicLinearPusher

print("Python executable:", sys.executable)

# Known simulator parameters.
K_A = 1.0       # Pusher stiffness [N/m]
H = 0.02       # Time step [s]
W = 0.1        # Minimum center separation [m]

# Numerical comparison threshold for this initial check.
VERIFICATION_TOL = 1e-7  # m

print("Imports successful.")

In [ ]:
class DrakePusher:
    """Evaluate one step of the original unsmoothed pusher."""

    def __init__(self, mass):
        self.system = QuasidynamicLinearPusher(
            k_a=K_A,
            m_o=float(mass),
            w=W,
            dt=H,
            dynamics_smoothing=0.0,
        )

    def step(self, x, u):
        context = self.system.CreateDefaultContext()

        context.SetDiscreteState(np.asarray(x, dtype=float))
        self.system.get_input_port().FixValue(
            context, np.array([float(u)])
        )

        update = self.system.EvalUniquePeriodicDiscreteUpdate(context)
        context.SetDiscreteState(update)

        return context.get_discrete_state_vector().CopyToVector()


def analytical_step(x, u, mass):
    """Predict [next pusher position, next object position]."""

    object_position = float(x[1])

    # Positive a means the command would push into the object.
    a = max(0.0, float(u) + W - object_position)

    alpha = K_A / (K_A + float(mass) / H**2)

    next_object = object_position + alpha * a

    if a > 0.0:
        next_pusher = next_object - W
    else:
        next_pusher = float(u)

    return np.array([next_pusher, next_object])


print("Drake adapter and analytical model ready.")

In [ ]:
masses = [0.005, 0.015, 0.030]  # kg
object_positions = [0.15, 0.20, 0.30]  # m

command_offsets = np.array([
    -0.020, -0.005, -0.0001,
     0.0,
     0.0001, 0.005, 0.010, 0.020,
])

records = []

for mass in masses:
    plant = DrakePusher(mass)

    for object_position in object_positions:
        # Start with a 0.02 m contact gap.
        x = np.array([
            object_position - W - 0.02,
            object_position,
        ])

        for offset in command_offsets:
            u = object_position - W + offset

            predicted = analytical_step(x, u, mass)
            simulated = plant.step(x, u)

            if not np.all(np.isfinite(simulated)):
                raise RuntimeError("Drake returned a nonfinite state.")

            records.append({
                "mass": mass,
                "object_position": object_position,
                "offset": offset,
                "predicted": predicted,
                "simulated": simulated,
                "error": np.max(np.abs(simulated - predicted)),
            })

errors = np.array([r["error"] for r in records])
worst_case = records[int(np.argmax(errors))]

print("Number of cases:", len(records))
print(f"Maximum state discrepancy: {errors.max():.6e} m")
print(f"Comparison threshold:     {VERIFICATION_TOL:.6e} m")
print("Within threshold:", bool(errors.max() <= VERIFICATION_TOL))
print("Worst-case mass [kg]:", worst_case["mass"])
print("Worst-case command offset [m]:", worst_case["offset"])
print("Analytical state:", worst_case["predicted"])
print("Drake state:     ", worst_case["simulated"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for mass in masses:
    selected = [
        r for r in records
        if r["mass"] == mass and r["object_position"] == 0.20
    ]

    offsets = np.array([r["offset"] for r in selected])
    predicted = np.array([r["predicted"][1] for r in selected])
    simulated = np.array([r["simulated"][1] for r in selected])

    line, = axes[0].plot(
        offsets,
        1000 * (predicted - 0.20),
        label=f"{1000 * mass:.0f} g",
    )
    axes[0].scatter(
        offsets,
        1000 * (simulated - 0.20),
        color=line.get_color(),
        marker="x",
    )

axes[0].set_xlabel("Command offset from contact onset [m]")
axes[0].set_ylabel("Object displacement [mm]")
axes[0].set_title("Analytical lines and Drake crosses")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(errors, ".")
axes[1].axhline(
    VERIFICATION_TOL,
    color="red",
    linestyle="--",
    label="Comparison threshold",
)
axes[1].set_xlabel("Validation case")
axes[1].set_ylabel("Maximum state discrepancy [m]")
axes[1].set_title("Numerical agreement")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Separate errors by state component and contact regime.
regimes = {
    "No contact": lambda a: a < -1e-12,
    "Contact onset": lambda a: abs(a) <= 1e-12,
    "Active contact": lambda a: a > 1e-12,
}

print(
    f"{'Regime':<18}"
    f"{'Cases':>7}"
    f"{'Pusher max [um]':>20}"
    f"{'Object max [um]':>20}"
)

for name, condition in regimes.items():
    selected = [r for r in records if condition(r["offset"])]

    component_errors = np.array([
        np.abs(r["simulated"] - r["predicted"])
        for r in selected
    ])

    print(
        f"{name:<18}"
        f"{len(selected):>7}"
        f"{1e6 * component_errors[:, 0].max():>20.6f}"
        f"{1e6 * component_errors[:, 1].max():>20.6f}"
    )

print("\nFive largest OBJECT-position discrepancies:")

ranked = sorted(
    records,
    key=lambda r: abs(r["simulated"][1] - r["predicted"][1]),
    reverse=True,
)

for r in ranked[:5]:
    object_error = r["simulated"][1] - r["predicted"][1]

    print(
        f"mass={1000*r['mass']:.1f} g, "
        f"object position={r['object_position']:.3f} m, "
        f"offset={r['offset']:.6f} m, "
        f"object error={1e6*object_error:.6f} um"
    )

In [ ]:
# Validate the operating range intended for identification.
probe_masses = np.linspace(0.005, 0.030, 11)
probe_positions = np.linspace(0.15, 0.30, 7)
probe_offsets = [0.005, 0.010, 0.015, 0.020]

probe_errors = []
worst_probe = None
max_probe_error = -1.0

for mass in probe_masses:
    plant = DrakePusher(mass)

    for object_position in probe_positions:
        x = np.array([
            object_position - W - 0.02,
            object_position,
        ])

        for offset in probe_offsets:
            u = object_position - W + offset

            simulated = plant.step(x, u)
            predicted = analytical_step(x, u, mass)

            if not np.all(np.isfinite(simulated)):
                raise RuntimeError("Nonfinite Drake state.")

            # Only object-position error: the quantity used by our estimator.
            error = abs(simulated[1] - predicted[1])
            probe_errors.append(error)

            if error > max_probe_error:
                max_probe_error = error
                worst_probe = (mass, object_position, offset)

print("Probe validation cases:", len(probe_errors))
print("Maximum object error [um]:", 1e6 * max_probe_error)
print("Worst case (mass, object position, offset):", worst_probe)

# An explicit engineering allowance for this prototype.
# This is not a certified global solver-error bound.
PROBE_NUMERICAL_ALLOWANCE = 5e-6  # 5 micrometers

print(
    "Within proposed 5 um allowance:",
    max_probe_error <= PROBE_NUMERICAL_ALLOWANCE,
)

In [ ]:
MASS_PRIOR = np.array([0.005, 0.030])  # kg

DISPLACEMENT_NOISE_BOUND = 10e-6       # 10 um
NUMERICAL_ALLOWANCE = 5e-6            # 5 um; empirical allowance
RESIDUAL_BOUND = (
    DISPLACEMENT_NOISE_BOUND + NUMERICAL_ALLOWANCE
)


def alpha_from_mass(mass):
    return K_A / (K_A + np.asarray(mass) / H**2)


def mass_from_alpha(alpha):
    return K_A * H**2 * (1.0 / np.asarray(alpha) - 1.0)


class MassEstimator:
    def __init__(self, mass_prior, residual_bound):
        m_low, m_high = mass_prior

        # Larger mass corresponds to smaller alpha.
        self.alpha_low = float(alpha_from_mass(m_high))
        self.alpha_high = float(alpha_from_mass(m_low))
        self.residual_bound = float(residual_bound)

    def update(self, excitation, measured_displacement):
        a = float(excitation)
        d = float(measured_displacement)

        if a <= 0:
            raise ValueError(
                "This experiment requires positive contact excitation."
            )

        observation_low = (d - self.residual_bound) / a
        observation_high = (d + self.residual_bound) / a

        new_low = max(self.alpha_low, observation_low)
        new_high = min(self.alpha_high, observation_high)

        if new_low > new_high:
            raise RuntimeError(
                "Empty parameter interval. Inspect the model "
                "and residual bound before continuing."
            )

        self.alpha_low = new_low
        self.alpha_high = new_high

    def mass_bounds(self):
        return np.array([
            mass_from_alpha(self.alpha_high),
            mass_from_alpha(self.alpha_low),
        ], dtype=float)

    def estimate(self):
        return float(np.mean(self.mass_bounds()))


print("Estimator ready.")
print("Total residual bound [um]:", RESIDUAL_BOUND * 1e6)

In [ ]:
# Used by the simulated plant and evaluation only.
TRUE_MASS = 0.015  # kg

plant = DrakePusher(TRUE_MASS)
estimator = MassEstimator(MASS_PRIOR, RESIDUAL_BOUND)
rng = np.random.default_rng(7)

x = np.array([0.0, 0.2])

# Predefined effective contact commands [m].
probe_sequence = [0.005, 0.010, 0.015, 0.020, 0.010, 0.020]

history = [estimator.mass_bounds()]
numerical_residuals = []
observations = []

for k, requested_excitation in enumerate(probe_sequence):
    # u is an absolute equilibrium-position command.
    u = x[1] - W + requested_excitation
    excitation = max(0.0, u + W - x[1])

    x_next = plant.step(x, u)
    if not np.all(np.isfinite(x_next)):
        raise RuntimeError("Nonfinite plant state.")

    actual_displacement = x_next[1] - x[1]

    # Evaluation only: this quantity is NOT passed to the estimator.
    predicted_true = analytical_step(x, u, TRUE_MASS)
    numerical_residual = x_next[1] - predicted_true[1]
    numerical_residuals.append(numerical_residual)

    if abs(numerical_residual) > NUMERICAL_ALLOWANCE:
        raise RuntimeError(
            "Numerical discrepancy exceeded the assumed allowance."
        )

    # Synthetic bounded displacement-measurement error.
    noise = rng.uniform(
        -DISPLACEMENT_NOISE_BOUND,
        DISPLACEMENT_NOISE_BOUND,
    )
    measured_displacement = actual_displacement + noise

    # Estimator receives only excitation and noisy displacement.
    estimator.update(excitation, measured_displacement)
    bounds = estimator.mass_bounds()
    history.append(bounds)

    observations.append({
        "step": k + 1,
        "excitation": excitation,
        "measured_displacement": measured_displacement,
        "mass_low": bounds[0],
        "mass_high": bounds[1],
    })

    print(
        f"Probe {k+1}: "
        f"a={1000*excitation:.1f} mm, "
        f"observed motion={1000*measured_displacement:.4f} mm, "
        f"mass interval=[{1000*bounds[0]:.3f}, "
        f"{1000*bounds[1]:.3f}] g"
    )

    x = x_next

history = np.asarray(history)

coverage = (
    (history[:, 0] <= TRUE_MASS)
    & (TRUE_MASS <= history[:, 1])
)

print("\nTrue mass [g]:", TRUE_MASS * 1000)
print("Final estimate [g]:", estimator.estimate() * 1000)
print("Final interval [g]:", history[-1] * 1000)
print("Truth inside every interval:", bool(np.all(coverage)))
print(
    "Maximum numerical residual during probes [um]:",
    1e6 * np.max(np.abs(numerical_residuals)),
)

assert np.all(coverage), "An estimated interval excluded the true mass."

In [ ]:
steps = np.arange(len(history))
widths_g = 1000 * (history[:, 1] - history[:, 0])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].fill_between(
    steps,
    history[:, 0] * 1000,
    history[:, 1] * 1000,
    alpha=0.25,
    label="Plausible mass interval",
)
axes[0].plot(
    steps,
    history.mean(axis=1) * 1000,
    "o-",
    label="Interval midpoint",
)
axes[0].axhline(
    TRUE_MASS * 1000,
    color="black",
    linestyle="--",
    label="True mass",
)
axes[0].set_xlabel("Number of probes")
axes[0].set_ylabel("Mass [g]")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(steps, widths_g, "o-")
axes[1].set_xlabel("Number of probes")
axes[1].set_ylabel("Mass interval width [g]")
axes[1].set_title("Remaining parameter uncertainty")
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate predictions on conditions not used during identification.
# These evaluations do not update the estimator.

initial_mass_estimate = float(np.mean(MASS_PRIOR))
learned_mass_estimate = estimator.estimate()
evaluation_plant = DrakePusher(TRUE_MASS)

evaluation_positions = [0.18, 0.24, 0.28]
evaluation_offsets = [0.006, 0.009, 0.013, 0.018]

before_errors = []
after_errors = []
true_displacements = []
before_predictions = []
after_predictions = []

for object_position in evaluation_positions:
    evaluation_x = np.array([
        object_position - W - 0.02,
        object_position,
    ])

    for offset in evaluation_offsets:
        u = object_position - W + offset

        actual_next = evaluation_plant.step(evaluation_x, u)
        before_next = analytical_step(
            evaluation_x, u, initial_mass_estimate
        )
        after_next = analytical_step(
            evaluation_x, u, learned_mass_estimate
        )

        before_errors.append(abs(before_next[1] - actual_next[1]))
        after_errors.append(abs(after_next[1] - actual_next[1]))

        true_displacements.append(actual_next[1] - object_position)
        before_predictions.append(before_next[1] - object_position)
        after_predictions.append(after_next[1] - object_position)

before_errors = np.asarray(before_errors)
after_errors = np.asarray(after_errors)

print("Held-out cases:", len(before_errors))
print("Initial mass estimate [g]:", 1000 * initial_mass_estimate)
print("Learned mass estimate [g]:", 1000 * learned_mass_estimate)
print(
    "Mean prediction error before learning [um]:",
    1e6 * before_errors.mean(),
)
print(
    "Mean prediction error after learning [um]:",
    1e6 * after_errors.mean(),
)
print(
    "Maximum prediction error after learning [um]:",
    1e6 * after_errors.max(),
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cases = np.arange(1, len(before_errors) + 1)

axes[0].plot(
    cases, 1000 * np.asarray(true_displacements),
    "ko-", label="Drake response",
)
axes[0].plot(
    cases, 1000 * np.asarray(before_predictions),
    "s--", label="Before learning",
)
axes[0].plot(
    cases, 1000 * np.asarray(after_predictions),
    "x--", label="After learning",
)
axes[0].set_xlabel("Held-out case")
axes[0].set_ylabel("Object displacement [mm]")
axes[0].legend()
axes[0].grid(True)

axes[1].bar(
    ["Before learning", "After learning"],
    [1e6 * before_errors.mean(), 1e6 * after_errors.mean()],
)
axes[1].set_ylabel("Mean absolute prediction error [um]")
axes[1].set_title("Prediction on new transitions")

plt.tight_layout()
plt.show()

In [ ]:
import os
# Notebook 1 — final export cell

from pathlib import Path

project_dir = Path(os.environ.get("PUSHER_PROJECT", Path.cwd().parent)).expanduser().resolve()
results_dir = project_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)

# The identification loop leaves its final state in x.
post_probe_state = np.asarray(x, dtype=float).copy()

assert post_probe_state.shape == (2,)
assert np.all(np.isfinite(post_probe_state))

print("State to export [pusher, object] [m]:", post_probe_state)
print("Expected object position for our current run: approximately 0.2021 m")

output_path = results_dir / "identification_result.npz"

np.savez(
    output_path,
    post_probe_state=post_probe_state,
    mass_prior=np.asarray(MASS_PRIOR),
    learned_mass=estimator.estimate(),
    mass_bounds=estimator.mass_bounds(),
    true_mass=TRUE_MASS,
    k_a=K_A,
    h=H,
    w=W,
)

print("Saved:", output_path)